## WELCOME TO WEEK 6

The Epic Finale Week

And

## WELCOME TO

# MCP

And welcome back to OpenAI Agents SDK ❤️❤️❤️

In [1]:
# The imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os

In [2]:
load_dotenv(override=True)

True

### Let's use MCP in OpenAI Agents SDK

1. Create a Client

2. Have it spawn a server

3. Collect the tools that the server can use

Let's try the Fetch mcp-server that we looked at last week

The below are depbug staments for windows: The most informative test will be checking the actual implementation of _make_subprocess_transport to see why it's raising NotImplementedError instead of providing an implementation on windows.

1) Check Python and asyncio versions

Create the orginal MCP's from Edd tutorial. 

In [ ]:
# import os
# import subprocess
# import sys
# import time

# # Get paths
# current_dir = os.getcwd()
# parent_dir = os.path.dirname(current_dir)
# script_abs_path = os.path.join(current_dir, "scripts", "mcp_runner.py")
# python_path = os.path.join(parent_dir, ".venv", "Scripts", "python.exe")

# # Print the command we're running
# command = [python_path, script_abs_path]
# print(f"Running command: {' '.join(command)}")

# try:
#     # Start the process
#     process = subprocess.Popen(
#         command,
#         stdout=subprocess.PIPE,
#         stderr=subprocess.PIPE,
#         text=True,
#         bufsize=1  # Line buffered
#     )
    
#     # Track start time for timeout
#     start_time = time.time()
#     timeout = 10  # 10 seconds
    
#     # Monitor output in real-time and enforce timeout
#     stdout_lines = []
#     stderr_lines = []
    
#     while process.poll() is None:  # While process is still running
#         # Check for timeout
#         if time.time() - start_time > timeout:
#             print(f"Process took longer than {timeout} seconds. Terminating...")
#             process.terminate()
#             break
            
#         # Check for output
#         if process.stdout.readable():
#             line = process.stdout.readline()
#             if line:
#                 print(f"OUT: {line.rstrip()}")
#                 stdout_lines.append(line)
        
#         if process.stderr.readable():
#             line = process.stderr.readline()
#             if line:
#                 print(f"ERR: {line.rstrip()}")
#                 stderr_lines.append(line)
        
#         time.sleep(0.1)  # Prevent CPU spinning
    
#     # Get any remaining output
#     remaining_stdout, remaining_stderr = process.communicate()
#     if remaining_stdout:
#         print(f"Remaining stdout: {remaining_stdout}")
#         stdout_lines.append(remaining_stdout)
#     if remaining_stderr:
#         print(f"Remaining stderr: {remaining_stderr}")
#         stderr_lines.append(remaining_stderr)
    
#     print(f"Process finished with return code: {process.returncode}")
    
# except Exception as e:
#     print(f"An error occurred: {e}")

In [ ]:
import os
import subprocess
import sys
import time

def run_mcp_server(server_type="fetch"):
    # Get paths
    current_dir = os.getcwd()
    parent_dir = os.path.dirname(current_dir)
    
    # Create scripts directory if it doesn't exist
    scripts_dir = os.path.join(current_dir, "scripts")
    os.makedirs(scripts_dir, exist_ok=True)
    
    # Choose the right script or create it on the fly
    if server_type == "fetch":
        script_name = "mcp_runner_fetch.py"
        script_content = """
# mcp_runner_fetch.py
import asyncio
import sys
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

async def main():
    load_dotenv(override=True)
    
    # Explicit setup of ProactorEventLoop for Windows
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}
    
    async with MCPServerStdio(params=fetch_params) as server:
        fetch_tools = await server.list_tools()
        
        for tool in fetch_tools:
            print(f"{tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())
        """
    elif server_type == "puppeteer":
        script_name = "mcp_runner_puppeteer.py"
        script_content = """
# mcp_runner_puppeteer.py
import asyncio
import sys
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

async def main():
    load_dotenv(override=True)
    
    # Explicit setup of ProactorEventLoop for Windows
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    puppeteer_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-puppeteer"]}
    
    async with MCPServerStdio(params=puppeteer_params) as server:
        puppeteer_tools = await server.list_tools()
        
        for tool in puppeteer_tools:
            print(f"{tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())
        """
    elif server_type == "filesystem":
        script_name = "mcp_runner_filesystem.py"
        script_content = """
# mcp_runner_filesystem.py
import asyncio
import sys
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

async def main():
    load_dotenv(override=True)
    
    # Explicit setup of ProactorEventLoop for Windows
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
    os.makedirs(sandbox_path, exist_ok=True)
    
    files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}
    
    async with MCPServerStdio(params=files_params) as server:
        file_tools = await server.list_tools()
        
        for tool in file_tools:
            print(f"{tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())
        """
    elif server_type == "playwright":
        script_name = "mcp_runner_playwright.py"
        script_content = """
# mcp_runner_playwright.py
import asyncio
import sys
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

async def main():
    load_dotenv(override=True)
    
    # Explicit setup of ProactorEventLoop for Windows
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    playwright_params = {"command": "npx", "args": ["@playwright/mcp@latest"]}
    
    async with MCPServerStdio(params=playwright_params) as server:
        playwright_tools = await server.list_tools()
        
        for tool in playwright_tools:
            print(f"{tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())
        """
    
    # Create or update the script file
    script_abs_path = os.path.join(scripts_dir, script_name)
    with open(script_abs_path, "w") as f:
        f.write(script_content)
    
    # Set up Python path
    python_path = os.path.join(parent_dir, ".venv", "Scripts", "python.exe")
    
    # Print the command we're running
    command = [python_path, script_abs_path]
    print(f"Running {server_type} MCP server with command: {' '.join(command)}")
    
    # Use subprocess.run with timeout instead of Popen with manual monitoring
    try:
        result = subprocess.run(
            command, 
            capture_output=True, 
            text=True, 
            timeout=15  # Increased timeout to 15 seconds
        )
        
        print(f"Return code: {result.returncode}")
        
        if result.stdout:
            print(f"Output from {server_type} MCP server:")
            print(result.stdout)
            return result.stdout
        else:
            print(f"No output produced from {server_type} MCP server.")
            return None
        
        if result.stderr:
            print(f"Errors from {server_type} MCP server:")
            print(result.stderr)
    
    except subprocess.TimeoutExpired:
        print(f"The {server_type} MCP server script took too long to run and was terminated.")
        return None
    except Exception as e:
        print(f"An error occurred with {server_type} MCP server: {e}")
        return None

# Test each MCP server
# fetch_output = run_mcp_server("fetch")
puppeteer_output = run_mcp_server("puppeteer")
filesystem_output = run_mcp_server("filesystem")
playwright_output = run_mcp_server("playwright")

# Now you can use the outputs to verify if each MCP server is working correctly

In [ ]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params) as server:
    fetch_tools = await server.list_tools()

for tool in fetch_tools:
    print(f"{tool.name}: {tool.description.replace('\n', ' ')}")

### And now repeat for 3 more!

In [ ]:
puppeteer_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-puppeteer"]}

async with MCPServerStdio(params=puppeteer_params) as server:
    puppeteer_tools = await server.list_tools()

for tool in puppeteer_tools:
    print(f"{tool.name}: {tool.description.replace('\n', ' ')}")


In [ ]:
puppeteer_tools[0]

In [ ]:

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params) as server:
    file_tools = await server.list_tools()

for tool in file_tools:
    print(f"{tool.name}: {tool.description.replace('\n', ' ')}")


In [ ]:

playwright_params = {"command": "npx", "args": ["@playwright/mcp@latest"]}

async with MCPServerStdio(params=playwright_params) as server:
    playwright_tools = await server.list_tools()

for tool in playwright_tools:
    print(f"{tool.name.replace('\n', ' ')}: {tool.description.replace('\n', ' ')}")


### And now.. bring on the Agent with Tools!

In [ ]:
# instructions = """
# You browse the internet to accomplish your instructions.
# You are highly capable at browsing the internet independently to accomplish your task, 
# including accepting all cookies and clicking 'not now' as
# appropriate to get to the content you need. If one website isn't fruitful, try another. 
# Be persistent until you have solved your assignment,
# trying different options and sites as needed.
# """


# async with MCPServerStdio(params=files_params, cache_tools_list=True) as mcp_server_files:
#     async with MCPServerStdio(params=playwright_params, cache_tools_list=True) as mcp_server_browser:
#         agent = Agent(
#             name="investigator", 
#             instructions=instructions, 
#             model="gpt-4o-mini",
#             mcp_servers=[mcp_server_files, mcp_server_browser]
#             )
#         with trace("investigate"):
#             result = await Runner.run(agent, "Find a great recipe for Banoffee Pie, then summarize it in markdown to banoffee.md")
#             print(result.final_output)


In [5]:
import os
import subprocess
import sys
import time

# Get paths
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
script_abs_path = os.path.join(current_dir, "scripts", "run_agent.py")
python_path = os.path.join(parent_dir, ".venv", "Scripts", "python.exe")

# Print the command we're running
command = [python_path, script_abs_path]
print(f"Running agent with command: {' '.join(command)}")

# Start the process with real-time monitoring
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1  # Line buffered
)

# Stream output in real-time
start_time = time.time()
timeout = 300  # 5 minutes timeout for the agent to complete its task

while process.poll() is None:  # While process is still running
    # Check for timeout
    if time.time() - start_time > timeout:
        print(f"Process took longer than {timeout} seconds. Terminating...")
        process.terminate()
        break
        
    # Check for output
    if process.stdout.readable():
        line = process.stdout.readline()
        if line:
            print(f"OUT: {line.rstrip()}")
    
    if process.stderr.readable():
        line = process.stderr.readline()
        if line:
            print(f"ERR: {line.rstrip()}")
    
    time.sleep(0.1)  # Prevent CPU spinning

# Get any remaining output
remaining_stdout, remaining_stderr = process.communicate()
if remaining_stdout:
    print(f"Remaining stdout: {remaining_stdout}")
if remaining_stderr:
    print(f"Remaining stderr: {remaining_stderr}")

print(f"Process finished with return code: {process.returncode}")

Running agent with command: c:\Users\Richa\Documents\projects\agents\.venv\Scripts\python.exe c:\Users\Richa\Documents\projects\agents\6_mcp\scripts\run_agent.py
OUT: Created run_agent.py script
Process finished with return code: 0


### Check out the trace

https://platform.openai.com/traces

### Now take a look at some MCP marketplaces

https://mcp.so

https://glama.ai/mcp

https://smithery.ai/

https://huggingface.co/blog/LLMhacker/top-11-essential-mcp-libraries

HuggingFace great community article:
https://huggingface.co/blog/Kseniase/mcp



